# Tiền xử lý EEG oddball — sub-01 đến sub-05

Toàn bộ code xử lý và kiểm tra nằm trong notebook này. Chạy từ trên xuống với kernel Python 3.11 và thư viện trong environment.yml.

Yêu cầu gốc: docs/03_analysis_protocol.md mục 1; tham số: configs/config.yaml. Vai trò và các bước cá nhân: reports/my_preprocessing_role_and_steps.md. Raw được giữ nguyên. ICA mất vài phút mỗi subject.



In [ ]:
import os
os.environ.setdefault('OMP_NUM_THREADS', '4')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '4')
os.environ.setdefault('MKL_NUM_THREADS', '4')
from pathlib import Path
ROOT = Path.cwd().resolve()
if not (ROOT / 'configs/config.yaml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'configs/config.yaml').exists(), 'Open from repository or notebooks directory'
print('Workspace:', ROOT)


## 1. Thư viện và cấu hình


In [ ]:
"""Reproducible oddball preprocessing, with original event timing and QC logs."""
import gc
import hashlib
import json
import logging
import re
import time
import warnings
import defusedxml.ElementTree  # required by MNE CapTrak reader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import mne
from mne_bids import BIDSPath, read_raw_bids
from mne_icalabel.iclabel import iclabel_label_components
import numpy as np
import pandas as pd
from scipy.signal import welch
from threadpoolctl import threadpool_limits
import yaml



CONFIG = yaml.safe_load((ROOT / 'configs/config.yaml').read_text(encoding='utf-8'))
print('Subjects:', CONFIG['dataset']['subjects'])


## 2. Bổ sung metadata EEG và tọa độ CapTrak

Chỉ tải các tệp nhỏ nếu thiếu, không tải thêm bản ghi EEG hay subject.


In [ ]:
import urllib.request
raw_root = ROOT / CONFIG['paths']['raw']
relative_files = ['task-oddball_eeg.json'] + [
    f'sourcedata/sub-{s}/coords/sub-{s}_captrak.bvct'
    for s in CONFIG['dataset']['subjects']
]
for relative in relative_files:
    destination = raw_root / relative
    if not destination.exists():
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(urllib.request.urlopen(
            'https://s3.amazonaws.com/openneuro.org/ds004621/' + relative,
            timeout=60).read())
print('Metadata and CapTrak coordinates are available.')


## 3. Theo dõi hội tụ ICA


In [ ]:
class ICAProgress(logging.Handler):
    """Read true iteration progress (Infomax n_iter can equal max on convergence)."""
    def __init__(self):
        super().__init__()
        self.step = 0
        self.change = float('inf')

    def emit(self, record):
        match = re.search(r'step (\d+) - .*wchange\s+([\d.eE+-]+)', record.getMessage())
        if match:
            self.step, self.change = int(match[1]), float(match[2])




## 4. Phát hiện kênh bất thường


In [ ]:
def bad_channel_scores(raw, cfg):
    """Flag flat channels or extreme log standard deviation (robust MAD z)."""
    data = raw.get_data()[:, ::cfg['sample_step']]
    std = np.std(data, axis=1)
    logs = np.log(np.maximum(std, np.finfo(float).tiny))
    mad = np.median(np.abs(logs - np.median(logs)))
    z = (logs - np.median(logs)) / max(1.4826 * mad, np.finfo(float).eps)
    frame = pd.DataFrame({'channel': raw.ch_names, 'std_uv': std * 1e6, 'robust_z': z})
    frame['auto_bad'] = (np.abs(z) > cfg['robust_z']) | (std * 1e6 < cfg['flat_std_uv'])
    return frame




## 5. Hình kiểm tra chất lượng


In [ ]:
def qc_plot(before, raw, scores, output, cfg):
    """Save signal, spectrum, and channel amplitude checks for visual review."""
    sfreq, original = before
    cleaned = raw.get_data(start=0, stop=int(cfg['duration_s'] * raw.info['sfreq']))
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    for data, rate, label in [(original, sfreq, 'Raw'), (cleaned, raw.info['sfreq'], 'Cleaned')]:
        f, power = welch(data, fs=rate, nperseg=min(int(rate * 2), data.shape[1]), axis=-1)
        axes[0, 0].plot(f, 10 * np.log10(np.maximum(np.median(power, axis=0) * 1e12, 1e-30)), label=label)
    axes[0, 0].set(xlim=(0, 100), xlabel='Frequency (Hz)', ylabel='Median PSD (dB uV²/Hz)')
    axes[0, 0].legend()
    axes[0, 1].bar(np.arange(len(scores)), scores['std_uv'], color=np.where(scores['auto_bad'], 'red', 'steelblue'))
    axes[0, 1].set(yscale='log', xlabel='Channel index', ylabel='Raw standard deviation (uV)')
    picks = [raw.ch_names.index(n) for n in ['Fp1', 'C3', 'Cz', 'Pz', 'O1']]
    for ax, data, rate, title in [(axes[1, 0], original, sfreq, 'Before'), (axes[1, 1], cleaned, raw.info['sfreq'], 'After')]:
        count = min(int(rate * cfg['trace_duration_s']), data.shape[1])
        for i in picks:
            ax.plot(np.arange(count) / rate, data[i, :count] * 1e6, label=raw.ch_names[i], linewidth=.6)
        ax.set(title=title, xlabel='Time (s)', ylabel='Amplitude (uV)')
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(output, dpi=cfg['dpi'])
    plt.close(fig)




## 6. Pipeline tiền xử lý một subject


In [ ]:
def preprocess(root, subject, config):
    """Process one full recording and save all three stimulus classes in FIF."""
    start = time.monotonic()
    pre = config['preprocessing']
    task = config['dataset']['task']
    rawroot = root / config['paths']['raw']
    out = root / config['paths']['derivatives']
    qc = root / 'figures' / 'preprocessing'
    out.mkdir(parents=True, exist_ok=True)
    qc.mkdir(parents=True, exist_ok=True)
    prefix = f'sub-{subject}_task-{task}'
    print(f'BEGIN {prefix}', flush=True)
    # BIDS trial_type only contains stimulus/response; recover event_type below.
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', message='Unable to map the following column')
        raw = read_raw_bids(BIDSPath(root=rawroot, subject=subject, task=task,
            datatype='eeg', suffix='eeg', extension='.vhdr'), verbose=False)
    raw.pick('eeg')
    raw.rename_channels({k: v for k, v in pre['channel_aliases'].items() if k in raw.ch_names})
    raw.set_montage(mne.channels.read_dig_captrak(rawroot / 'sourcedata' / f'sub-{subject}' / 'coords' / f'sub-{subject}_captrak.bvct'), on_missing='raise')
    table = pd.read_csv(rawroot / f'sub-{subject}' / 'eeg' / f'{prefix}_events.tsv', sep='\t')
    descriptions = ['Stimulus/' + v for v in table['event_type']]
    raw.set_annotations(mne.Annotations(table['onset'].to_numpy(), np.zeros(len(table)), descriptions))
    raw.load_data()
    if not np.isfinite(raw.get_data()).all():
        raise ValueError('Non-finite raw samples')
    before = (raw.info['sfreq'], raw.get_data(start=0, stop=int(pre['qc']['duration_s'] * raw.info['sfreq'])))
    scores = bad_channel_scores(raw, pre['bad_channels'])
    auto = scores.loc[scores.auto_bad, 'channel'].tolist()
    bads = sorted(set(raw.info['bads'] + auto + pre['bad_channels']['manual'].get(subject, [])))
    raw.info['bads'] = bads
    scores.to_csv(qc / f'{prefix}_channels.csv', index=False)
    raw.filter(pre['l_freq'], pre['h_freq'], method='fir', phase='zero', verbose=False)
    # 50 Hz is already outside the 40 Hz low-pass; no unnecessary notch.
    if bads:
        raw.interpolate_bads(reset_bads=True, mode='accurate', method={'eeg': 'spline'}, verbose=False)
    raw.set_eeg_reference(pre['reference'], projection=False, verbose=False)
    fit_raw = raw.copy().filter(pre['ica']['fit_l_freq'], None, verbose=False)
    ica = mne.preprocessing.ICA(n_components=pre['ica']['n_components'], method=pre['ica']['method'],
        fit_params={'extended': pre['ica']['extended'], 'w_change': pre['ica']['w_change'], 'verbose': True}, max_iter=pre['ica']['max_iter'], random_state=config['seed'])
    print(f'{prefix}: ICA fit, bad channels={bads}', flush=True)
    progress = ICAProgress()
    mne.utils.logger.addHandler(progress)
    try:
        ica.fit(fit_raw, decim=pre['ica']['decim'], verbose=True)
    finally:
        mne.utils.logger.removeHandler(progress)
    if not progress.step or (progress.step >= pre['ica']['max_iter'] and progress.change >= pre['ica']['w_change']):
        raise RuntimeError('ICA convergence not verified from iteration log')
    probabilities = iclabel_label_components(fit_raw, ica, backend='onnx')
    names = ['brain', 'muscle artifact', 'eye blink', 'heart beat', 'line noise', 'channel noise', 'other']
    labels = pd.DataFrame(probabilities, columns=names)
    labels.index.name = 'component'
    reject_cols = pre['ica']['iclabel_reject']
    labels['excluded'] = labels[reject_cols].max(axis=1) >= pre['ica']['iclabel_threshold']
    ica.exclude = labels.index[labels.excluded].tolist()
    labels.to_csv(qc / f'{prefix}_iclabel.csv')
    ica.save(out / f'{prefix}_ica.fif', overwrite=True, verbose=False)
    if ica.exclude:
        figs = ica.plot_components(picks=ica.exclude, show=False)
        if not isinstance(figs, list): figs = [figs]
        for i, fig in enumerate(figs):
            fig.savefig(qc / f'{prefix}_excluded_ics_{i}.png', dpi=pre['qc']['dpi'])
            plt.close(fig)
    del fit_raw
    gc.collect()
    ica.apply(raw, verbose=False)
    event_id = {name: i + 1 for i, name in enumerate(['standard', 'target', 'distractor'])}
    mapped = {config['dataset']['events'][k]: v for k, v in event_id.items()}
    events, _ = mne.events_from_annotations(raw, event_id=mapped, verbose=False)
    original_onsets = events[:, 0] / raw.info['sfreq']
    breaks = table.loc[table.event_type == 'S 10', 'onset'].to_numpy()
    blocks = np.searchsorted(breaks, original_onsets, side='right')
    raw, events = raw.resample(pre['resample_sfreq'], events=events, verbose=False)
    qc_plot(before, raw, scores, qc / f'{prefix}_qc.png', pre['qc'])
    ep = pre['epoch']
    epochs = mne.Epochs(raw, events, event_id, tmin=ep['tmin'], tmax=ep['tmax'], baseline=tuple(ep['baseline']),
        reject={'eeg': ep['reject_ptp_uv'] * 1e-6}, preload=True,
        metadata=pd.DataFrame({'subject': subject, 'onset_s': original_onsets, 'block': blocks}), verbose=False)
    epochs.drop_bad(verbose=False)
    counts_before = {k: int(np.sum(events[:, 2] == v)) for k, v in event_id.items()}
    counts_after = {k: len(epochs[k]) for k in event_id}
    excluded = counts_after['target'] / counts_before['target'] < pre['min_target_retained']
    epochs.save(out / f'{prefix}_epo.fif', overwrite=True, verbose=False)
    if not np.isfinite(epochs.get_data()).all(): raise ValueError('Non-finite epochs')
    row = {'subject': f'sub-{subject}', 'n_channels': len(raw.ch_names), 'sfreq': epochs.info['sfreq'],
        'bad_channels': ';'.join(bads), 'n_bad_channels': len(bads), 'n_ica': int(ica.n_components_),
        'ica_iterations': progress.step, 'ica_final_weight_change': progress.change,
        'n_ic_removed': len(ica.exclude), 'excluded_subject': excluded,
        'exclusion_reason': 'target_retention_below_threshold' if excluded else '',
        'montage': 'subject CapTrak', 'notch_applied': False, 'visual_review': 'pending',
        'seconds': round(time.monotonic() - start, 1)}
    for k in event_id:
        row.update({f'{k}_before': counts_before[k], f'{k}_after': counts_after[k], f'{k}_retained': counts_after[k] / counts_before[k]})
    (out / f'{prefix}_summary.json').write_text(json.dumps(row, indent=2), encoding='utf-8')
    print('DONE ' + json.dumps(row), flush=True)
    return row




## 7. Hàm chạy và ghi log/provenance


In [ ]:
def run(root, subjects=None, overwrite=False):
    """Run selected configured subjects; resume completed subjects by default."""
    config_path = root / 'configs/config.yaml'
    config = yaml.safe_load(config_path.read_text(encoding='utf-8'))
    subjects = subjects or config['dataset']['subjects']
    out = root / config['paths']['derivatives']
    rows = []
    for subject in subjects:
        summary = out / f'sub-{subject}_task-{config["dataset"]["task"]}_summary.json'
        if summary.exists() and not overwrite:
            rows.append(json.loads(summary.read_text()))
        else:
            with threadpool_limits(limits=4):
                rows.append(preprocess(root, subject, config))
        pd.DataFrame(rows).to_csv(root / 'results/preproc_log.csv', index=False)
        gc.collect()
    provenance = {'config_sha256': hashlib.sha256(config_path.read_bytes()).hexdigest(),
        'config': config, 'mne_version': mne.__version__, 'subjects': subjects,
        'notes': ['ICA fitted on full unlabelled recording', 'ICA uses 1-40 Hz per protocol; ICLabel was trained on 1-100 Hz',
                  'ICA fitting decimates filtered data by 4; final recording resampled after ICA',
                  'Five-subject exploratory subset; visual QC required before final analysis']}
    (out / 'provenance.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')


## 8. Thực hiện cho 5 subject

Mặc định bỏ qua subject đã có summary. Đặt overwrite=True nếu cần chạy lại. Kiểm tra hình QC trước khi chấp nhận kết quả.


In [ ]:
run(ROOT, subjects=CONFIG["dataset"]["subjects"], overwrite=False)


## 9. Kiểm tra dữ liệu đã lưu

Đọc lại FIF và kiểm tra 250 Hz, 251 mẫu, baseline, 3 lớp, peak-to-peak, thứ tự thời gian và 5 block.


In [ ]:
root = ROOT
config = CONFIG
pre = config["preprocessing"]
results = []
for subject in config['dataset']['subjects']:
    name = f'sub-{subject}_task-{config["dataset"]["task"]}'
    path = root / config['paths']['derivatives'] / f'{name}_epo.fif'
    epochs = mne.read_epochs(path, preload=True, verbose=False)
    data = epochs.get_data()
    assert epochs.info['sfreq'] == pre['resample_sfreq']
    assert np.allclose([epochs.tmin, epochs.tmax], [pre['epoch']['tmin'], pre['epoch']['tmax']])
    assert set(epochs.event_id) == {'standard', 'target', 'distractor'}
    assert np.isfinite(data).all()
    assert not epochs.info['bads']
    assert data.shape[-1] == round((epochs.tmax - epochs.tmin) * epochs.info['sfreq']) + 1
    assert np.max(np.ptp(data, axis=-1)) <= pre['epoch']['reject_ptp_uv'] * 1e-6 * 1.00001
    baseline = (epochs.times >= pre['epoch']['baseline'][0]) & (epochs.times <= pre['epoch']['baseline'][1])
    assert np.max(np.abs(data[:, :, baseline].mean(axis=-1))) < 1e-10
    assert list(epochs.metadata.columns) == ['subject', 'onset_s', 'block']
    assert epochs.metadata['onset_s'].is_monotonic_increasing
    assert set(epochs.metadata['block']) <= set(range(config['evaluation']['within']['n_splits']))
    results.append({'subject': subject, 'epochs': len(epochs), 'channels': len(epochs.ch_names),
                    'samples': len(epochs.times), 'blocks': sorted(epochs.metadata.block.unique().tolist()), 'verified': True})
    print(results[-1], flush=True)
(root / 'results/preproc_verification.json').write_text(json.dumps(results, indent=2), encoding='utf-8')


## 10. Xem thống kê và hình QC


In [ ]:
from IPython.display import display, Image
# Kiểm tra epoch trên toàn bộ thời gian thí nghiệm, không chỉ đoạn đầu recording.
qc_dir = ROOT / 'figures/preprocessing'
for subject in CONFIG['dataset']['subjects']:
    prefix = f'sub-{subject}_task-oddball'
    epochs = mne.read_epochs(ROOT / CONFIG['paths']['derivatives'] / f'{prefix}_epo.fif', preload=True, verbose=False)
    amplitudes = np.ptp(epochs.get_data(), axis=-1) * 1e6
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    image = axes[0].imshow(amplitudes.T, origin='lower', aspect='auto',
                           vmin=0, vmax=CONFIG['preprocessing']['epoch']['reject_ptp_uv'])
    axes[0].set(xlabel='Retained trial index', ylabel='EEG channel index', title=f'{prefix}: peak-to-peak')
    fig.colorbar(image, ax=axes[0], label='Amplitude (uV)')
    for condition in ['standard', 'target', 'distractor']:
        if len(epochs[condition]):
            evoked = epochs[condition].average()
            axes[1].plot(evoked.times * 1000, evoked.get_data(picks=['Pz'])[0] * 1e6, label=condition)
    axes[1].axvline(0, color='gray', linewidth=.6)
    axes[1].set(xlabel='Time (ms)', ylabel='Amplitude at Pz (uV)', title='ERP quality check (exploratory)')
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(qc_dir / f'{prefix}_epoch_qc.png', dpi=CONFIG['preprocessing']['qc']['dpi'])
    plt.close(fig)
    del epochs

display(pd.read_csv(ROOT / 'results/preproc_log.csv'))
for subject in CONFIG['dataset']['subjects']:
    for path in sorted(qc_dir.glob(f'sub-{subject}_task-oddball_*.png')):
        print(path.name)
        display(Image(filename=str(path)))


## Ghi nhận kiểm tra bằng mắt
Kết quả lần chạy này đã được Codex xem các hình QC tổng hợp, chưa thay cho kiểm tra tương tác từng kênh. Log ghi `summary_plots_reviewed`. Nếu chạy lại với `overwrite=True`, trạng thái trở về `pending` và phải xem lại hình.
